## 0)  Project Root

In [22]:
import sys
from pathlib import Path


def get_project_root(project_dir_name="Project", marker=".git"):
    """
    1) Walk upwards until we find the repo root (contains .git).
    2) Return <repo_root>/<project_dir_name> as the actual project root
       (the folder that contains models/, training/, notebooks/, etc.).
    3) Add that project root to sys.path for imports.
    """
    current = Path.cwd().resolve()

    # Step 1: find repo root by .git
    repo_root = None
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            repo_root = parent
            break

    if repo_root is None:
        raise RuntimeError(f"Repo root not found (no '{marker}' directory)")

    # Step 2: define actual project root
    project_root = repo_root / project_dir_name
    if not project_root.exists():
        raise RuntimeError(
            f"Found repo root at {repo_root}, but '{project_dir_name}' folder not found."
        )

    # Step 3: add to sys.path
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    print(f"[OK] Repo root:    {repo_root}")
    print(f"[OK] Project root: {project_root}")
    return project_root

PROJECT_ROOT = get_project_root()

[OK] Repo root:    /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning
[OK] Project root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project


In [23]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Evaluating on:", device)

Evaluating on: mps


In [24]:
from pathlib import Path
import json
import torch


SEQ_DIR = PROJECT_ROOT / "03_Sequences"
RUN_DIR = PROJECT_ROOT / "runs" / "LSTMClassifier_bs16_20251219-071043_UTC"  # <- pick one


CONFIG_PATH = RUN_DIR / "config.json"
CKPT_PATH   = RUN_DIR / "best_model.pt"


In [25]:
from utils.data_utils import make_test_loader

test_loader, X_test_raw, y_test, config = make_test_loader(
    run_dir=RUN_DIR,
    seq_dir=SEQ_DIR,
    shuffle=False
)

model_class_name = config["model_class"]
model_kwargs     = config["model_kwargs"]

In [26]:
from models import LSTMClassifier  # add more as you create them

MODEL_REGISTRY = {
    "LSTMClassifier": LSTMClassifier,
    # "GRUClassifier": GRUClassifier,
    # "TransformerClassifier": TransformerClassifier,
}

ModelClass = MODEL_REGISTRY[model_class_name]
model = ModelClass(**model_kwargs).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()


LSTMClassifier(
  (lstm): LSTM(15, 64, num_layers=2, batch_first=True, dropout=0.1)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

In [27]:
import numpy as np
from tqdm import tqdm

all_probs  = []
all_preds  = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in tqdm(test_loader, desc="Predicting (test)", leave=False):
        X_batch = X_batch.to(device)

        logits = model(X_batch).view(-1)                 # ensure shape (batch,)
        probs  = torch.sigmoid(logits)                   # UP probability in [0,1]
        preds  = (probs >= 0.5).long()

        all_probs.extend(probs.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

all_probs  = np.array(all_probs).reshape(-1)
all_preds  = np.array(all_preds).reshape(-1).astype(int)
all_labels = np.array(all_labels).reshape(-1).astype(int)


In [28]:
meta = json.loads((SEQ_DIR / "meta.json").read_text())

t_to_end_min_idx = meta["feature_cols"].index("t_to_end_min")
t_to_end_min_values = X_test_raw[:, -1, t_to_end_min_idx].astype(int)

assert len(t_to_end_min_values) == len(all_labels) == len(all_probs)


In [29]:
import pandas as pd

df = pd.DataFrame({
    "t_to_end_min": t_to_end_min_values,
    "y_true": all_labels,
    "p_up": all_probs,
    "y_pred": all_preds,
})


In [30]:
df["tp"] = ((df.y_true==1) & (df.y_pred==1)).astype(int)
df["fp"] = ((df.y_true==0) & (df.y_pred==1)).astype(int)
df["tn"] = ((df.y_true==0) & (df.y_pred==0)).astype(int)
df["fn"] = ((df.y_true==1) & (df.y_pred==0)).astype(int)
df["correct"] = (df.y_true == df.y_pred).astype(int)

stats = df.groupby("t_to_end_min").agg(
    count=("correct","count"),
    tp=("tp","sum"),
    fp=("fp","sum"),
    tn=("tn","sum"),
    fn=("fn","sum"),
    accuracy=("correct","mean"),
    avg_p_up=("p_up","mean"),
    base_rate=("y_true","mean"),
).reset_index()

stats["accuracy_pct"] = stats["accuracy"] * 100
print(stats.to_string(index=False))


 t_to_end_min  count    tp   fp    tn   fn  accuracy  avg_p_up  base_rate  accuracy_pct
            1  31564 15206  138 15685  535  0.978678  0.470371   0.498701     97.867824
            2  31564 14568  851 14972 1173  0.935876  0.469916   0.498701     93.587631
            3  31564 14111 1444 14379 1630  0.902611  0.471042   0.498701     90.261057
            4  31564 13696 1920 13903 2045  0.874382  0.471372   0.498701     87.438221
            5  31564 13347 2423 13400 2394  0.847389  0.473618   0.498701     84.738943
            6  31564 12981 2924 12899 2760  0.819921  0.477012   0.498701     81.992143
            7  31564 12749 3359 12464 2992  0.798790  0.481146   0.498701     79.878976
            8  31564 12372 3856 11967 3369  0.771100  0.485290   0.498701     77.109999
            9  31564 11985 4372 11451 3756  0.742491  0.489148   0.498701     74.249145
           10  31564 11570 4979 10844 4171  0.710113  0.493709   0.498701     71.011279
           11  31564 11238 5540 